## Part 1: Environment Setup

In [ ]:
# Install required packages
# Uncomment to install if needed:
# !pip install langchain langchain-openai langchain-community pypdf faiss-cpu python-dotenv langsmith

In [ ]:
# Verify langchain is installed
# TODO: Add command to check langchain version

In [2]:
# TODO: Import necessary libraries
# Import: os, dotenv, PyPDFLoader, RecursiveCharacterTextSplitter,
# OpenAIEmbeddings, ChatOpenAI, FAISS, ChatPromptTemplate, StrOutputParser

import os
from dotenv import load_dotenv
# Add remaining imports here
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
load_dotenv()

# Set API Keys
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

print("✅ Libraries imported successfully!")

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported successfully!


## Part 2: Loading PDF Documents

### Understanding PDF Loading
- We'll use PyPDFLoader to extract text from PDF files
- Each page becomes a separate document with metadata
- This allows us to track which page information came from

### Key Questions:
1. Why is metadata important when loading PDFs?
2. What information does metadata contain?
3. How would you handle PDFs with different formats or encryption?

In [4]:
# TODO: Load a PDF file
# 1. Set the pdf_path to your PDF file
# 2. Create a PyPDFLoader
# 3. Load the documents
# 4. Print number of pages loaded
# 5. Display first page preview and metadata

pdf_path = "D:/Mentoring/learwithsarvesh/12weekcrash Course/Week4 - PDF Document Summarizer (RAG Project)/HDFC-Life-Group-Term-Life-Policy.pdf"

## Step 2:
loader = PyPDFLoader(pdf_path)

#Step 3:
documents = loader.load()

# Step 4:
print(f"Loaded {len(documents)} pages from pdf")

# Step 5: 
print(f" First Page preview: \n{documents[0].page_content[:300]}...")

Loaded 30 pages from pdf
 First Page preview: 
F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                        
 
 
 
 
 
   HDFC Life Group Term Life 
 
OF 
 
 
«OWNERNAME» 
 
 
 
 
 
  
Based on the Proposal and the declarations and 
any 
statement made or referred to therein, 
We will pay the Benefits mentione...


## Part 3: Text Chunking

### Why Chunking?
- LLMs have token limits for context
- Embeddings work better with focused, manageable chunks
- Better retrieval accuracy with smaller, semantic units

### Chunking Strategy:
- **chunk_size**: Size of each text chunk (characters)
- **chunk_overlap**: Overlap between chunks to maintain context
- **RecursiveCharacterTextSplitter**: Splits intelligently at paragraph/sentence boundaries

### Experiment:
- What happens if chunk_size = 500? Too small?
- What happens if chunk_overlap = 0? Lost context?
- Try adjusting these values and observe the results

- Smaller Chunk = better retrievel -> Not always correct | Context Matters

In [9]:
# TODO: Create a text splitter and split documents
# 1. Initialize RecursiveCharacterTextSplitter with chunk_size=1000, chunk_overlap=200
# 2. Split the documents into chunks
# 3. Print number of chunks created
# 4. Display first chunk content and metadata

# Write your code here

text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n",  ## Paragraph
                  "\n",    ## New Line
                  " ",     ## New Word
                  ""],    ## Next Character
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len,
)

## Step 2: Split the docs into Chunks:
chunks = text_splitter.split_documents(documents)

## Step 3: Print number of chunks created
print(f"Created {len(chunks)} chunks from {len(documents)} pages")

## Step 4: Display first chunk content and metadata
print(f"\n Sample Chunk: \n {chunks[0].page_content}")
print(f"\n --------------------")
print(f"\n Chunk Metadata: {chunks[0].metadata}")


Created 115 chunks from 30 pages

 Sample Chunk: 
 F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                        
 
 
 
 
 
   HDFC Life Group Term Life 
 
OF 
 
 
«OWNERNAME» 
 
 
 
 
 
  
Based on the Proposal and the declarations and 
any 
statement made or referred to therein, 
We will pay the Benefits mentioned in this Policy 
subject to the terms and conditions contained 
herein 
 
 
 
 
 
 
<< Designation of the Authorised Signatory >>

 --------------------

 Chunk Metadata: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2023-08-24T19:47:11+05:30', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions', 'author': 'Atul Bhatia', 'moddate': '2023-08-24T19:47:11+05:30', 'source': 'D:/Mentoring/learwithsarvesh/12weekcrash Course/Week4 - PDF Document Summarizer (RAG Project)/HDFC-Life-Group-Term-Life-Policy.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1'}


16k context window -> 20k context to LLM call -> LLM will refuse --> Throws Errors

In [10]:
# TODO: Analyze chunk statistics
# 1. Calculate min, max, and average chunk sizes
# 2. Print statistics to understand chunk distribution

## Step 1: Calculate min, max, and average chunk sizes
chunk_lengths = [len(i.page_content) for i in chunks]

print(" Chunk Stats")
print(f" - Minimum chunk size : {min(chunk_lengths)} characters")
print(f" - Maximum chunk size : {max(chunk_lengths)} characters")
print(f" - Average chunk size : {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
print(f"\n ==============This is the Stats of the chunks==============")


 Chunk Stats
 - Minimum chunk size : 232 characters
 - Maximum chunk size : 999 characters
 - Average chunk size : 849 characters

 ==============This is the Stats of the chunks==============


## Part 4: Creating Embeddings

### What are Embeddings?
- Vector representations of text that capture semantic meaning
- Similar texts have similar vectors (close in vector space)
- Enable semantic search (find meaning, not just keywords)

### OpenAI Embeddings:
- Model: `text-embedding-3-small` (1536 dimensions)
- Converts text → numerical vectors
- These vectors are stored in the vector database

### Think About:
1. What does "1536 dimensions" mean?
2. Why do similar sentences have similar embeddings?
3. How would cosine similarity help us find relevant chunks?

Health Repots:
1. Bp/ Blood Sugar/ Cholestrol/ HB/ Vitamins -> 10k

5 -> 5k

gpt 3.5-turbo
gpt 5.2
gpt 4o-mini
gpt 4o

text-embedddings-3-small 

In [15]:
# TODO: Initialize OpenAI Embeddings
# 1. Create OpenAIEmbeddings with model="text-embedding-3-small"
# 2. Test with a sample text
# 3. Print embedding dimension and first 5 values

# Step 1. Create OpenAIEmbeddings with model="text-embedding-3-small"
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small"
    #model = "gpt-3.5-turbo-0125"
)

# Step 2. Test with a sample text
sample_text = "What is artificial intelligence?"
sample_embedding = embeddings.embed_query(sample_text)

# Step 3. Print embedding dimension and first 5 values
print(f" Embeddings initialized successfully")
print(f" Embedding Dimension: {len(sample_embedding)}")
print(f" First 5 values: {sample_embedding[:5]}")
print(f"\n Each chunk will be converted to a {len(sample_embedding)} - dimension vector")


 Embeddings initialized successfully
 Embedding Dimension: 1536
 First 5 values: [0.006162669975310564, -0.01451562624424696, -0.03586096316576004, 0.0057395463809370995, 0.021922778338193893]

 Each chunk will be converted to a 1536 - dimension vector


In [12]:
# TODO: Initialize OpenAI Embeddings
# 1. Create OpenAIEmbeddings with model="text-embedding-3-small"
# 2. Test with a sample text
# 3. Print embedding dimension and first 5 values

# Step 1. Create OpenAIEmbeddings with model="text-embedding-3-small"
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-large"
)

# Step 2. Test with a sample text
sample_text = "What is artificial intelligence?"
sample_embedding = embeddings.embed_query(sample_text)

# Step 3. Print embedding dimension and first 5 values
print(f" Embeddings initialized successfully")
print(f" Embedding Dimension: {len(sample_embedding)}")
print(f" First 5 values: {sample_embedding[:5]}")
print(f"\n Each chunk will be converted to a {len(sample_embedding)} - dimension vector")


 Embeddings initialized successfully
 Embedding Dimension: 3072
 First 5 values: [-0.008916644379496574, -0.0029415732715278864, -0.017454100772738457, -0.001493767718784511, 0.02350960485637188]

 Each chunk will be converted to a 3072 - dimension vector


### Demonstrating Semantic Similarity

**Question:** What should happen when we compare "The cat sat on the mat" vs "A feline rested on the rug"?

In [17]:
import numpy as np

# TODO: Calculate cosine similarity between sentence pairs
# 1. Create a function to calculate cosine similarity
# 2. Compare similar sentences ("cat on mat" vs "feline on rug")
# 3. Compare dissimilar sentences ("cat on mat" vs "Python programming")
# 4. Print and discuss the results

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

text1 = "The cat sat on the wall"
text2 = "Cat likes to chase mice"
text3 = "Python is a programming language"

embed1 = embeddings.embed_query(text1)
embed2 = embeddings.embed_query(text2)
embed3 = embeddings.embed_query(text3)

similarity_1_2 = cosine_similarity(embed1, embed2)
similarity_1_3 = cosine_similarity(embed1, embed3)


print("Semantic Similarity")
print(f"\n {text1}")
print(f"\n {text2}")
print(f"Similarity: {similarity_1_2:.4f}")
print(f"Similarity: {similarity_1_3:.4f}")


Semantic Similarity

 The cat sat on the wall

 Cat likes to chase mice
Similarity: 0.4611
Similarity: 0.1144


## Cosine Similarity:
Cosine Similarity measures how similar two pieces of text are by checking whether their meanings point towards same direction

- It doen't care for exact words
- Cosine means angle between 2 vectors
- Different Direction --> diff meaning

## Part 5: Building FAISS Vector Store

### What is FAISS?
- **Facebook AI Similarity Search**
- Efficient library for similarity search
- Stores embeddings and finds nearest neighbors
- Perfect for RAG applications

### How it Works:
1. Convert all chunks to embeddings
2. Store in FAISS index
3. Given a query, find most similar chunks
4. Return relevant context to the LLM

### Discussion:
- Why is an index better than comparing every chunk?
- What happens when you have 1 million documents?

1. FAISS - Index: Like a librar catelog. Instead of checking complete book, you look up the catelog

In [18]:
# TODO: Create FAISS vector store
# 1. Create a FAISS vector store from chunks and embeddings
# 2. Print confirmation message with number of chunks stored

print("⏳ Creating vector store... This may take a moment...")

vectorstore = FAISS.from_documents(
    documents = chunks,
    embedding = embeddings
)

print(f"Vector store created successfully")
print(f" Stored {len(chunks)} chunk embedddings")
print(f" Ready for semantic search")

⏳ Creating vector store... This may take a moment...
Vector store created successfully
 Stored 115 chunk embedddings
 Ready for semantic search


### Testing the Vector Store - Similarity Search

In [22]:
# TODO: Test similarity search
# 1. Create a test query
# 2. Search for top 3 similar chunks
# 3. Display the results with metadata

test_query = "What are the main topics discussed in this document?"

## Step2: Search for top 3 similar chunks
relevent_docs = vectorstore.similarity_search(test_query, k = 5)

print(f" Query: {test_query}")
print(f"Found {len(relevent_docs)} relevent_chunks:\n")

for i, doc in enumerate(relevent_docs, 1):
    print(f"Chunk {i} (Page {doc.metadata.get('page', 'N/A')}):")
    print(doc.page_content[:300] + "...")

 Query: What are the main topics discussed in this document?
Found 5 relevent_chunks:

Chunk 1 (Page 7):
15. Benefits means the Benefit as mentioned in Part C of this Policy Document. 
 
16. Benefit Expiry Age means the Age in Years last birthday as mentioned in the Policy Schedule. 
 
17. Certificate of Insurance in respect of an Insured Member , means the Certificate of Insurance 
issued by the Compa...
Chunk 2 (Page 1):
purchased HDFC Life Insurance Policy:  
 
 Policy Schedule   :  Summary of key features of your HDFC Life Insurance Policy 
 Premium Receipt   :  Acknowledgement of the first Premium paid by you 
 Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                        ...
Chunk 3 (Page 24):
(c) disputes over Premium paid or payable in terms of insurance Policy; 
(d) misrepresentation of Policy terms and conditions at any time in the Policy document or Policy contract; 
(e) legal construction of insurance policies in so far as the disput

In [21]:
# TODO: Display similarity scores
# 1. Use similarity_search_with_score to get confidence scores
# 2. Print the scores for each chunk
# 3. Discuss what the scores mean

results_with_score = vectorstore.similarity_search_with_score(test_query, k = 3)

print(f" Similarity Score (lower = more similar): \n")
for i, (doc, score) in enumerate(results_with_score, 1):
    print(f"Chunk {i}: Score = {score:.4f} | page {doc.metadata.get('page', 'N/A')}")

print(f"\n These scores help us understand retrievel qualities!")

 Similarity Score (lower = more similar): 

Chunk 1: Score = 1.4221 | page 7
Chunk 2: Score = 1.4344 | page 1
Chunk 3: Score = 1.4586 | page 24

 These scores help us understand retrievel qualities!


### Save and Load Vector Store (Optional)

In [23]:
# TODO: Save vector store locally
# 1. Save the vector store to disk
# 2. Optionally load it back


vectorstore.save_local("pdf_vectorstore")

## Part 6: Building the RAG Pipeline

### RAG Architecture:
```
User Query → Retrieve Relevant Chunks → Feed to LLM → Generate Answer
```

### Components:
1. **Retriever**: Gets relevant chunks from vector store
2. **LLM**: Generates answer based on retrieved context
3. **Prompt**: Instructs LLM how to use the context
4. **Chain**: Connects everything together

### Challenge:
- What should you tell the LLM about the context?
- How can you prevent the LLM from making up answers?

In [24]:
# TODO: Initialize LLM and Retriever
# 1. Create ChatOpenAI instance with model="gpt-3.5-turbo" and temperature=0
# 2. Create a retriever from the vector store with k=3
# 3. Print confirmation messages

## Step 1: Create ChatOpenAI instance with model="gpt-3.5-turbo" and temperature=0
llm = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature = 0
)

## Step 2. Create a retriever from the vector store with k=3
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {'k': 3}
)

## Step 3. Print confirmation messages
print(f" LLM and Retriever initialized!")
print(f" Retriever will fetch {retriever.search_kwargs['k']} relevent chunks per query")

 LLM and Retriever initialized!
 Retriever will fetch 3 relevent chunks per query


### Create a Custom Prompt Template

**Your Turn:** Design a prompt that:
1. Uses the retrieved context
2. Instructs the model to say "I don't know" if context isn't helpful
3. Asks for concise but informative answers

In [27]:
# TODO: Create prompt template
# 1. Design system prompt with instructions
# 2. Create ChatPromptTemplate with system and human messages
# 3. Print the prompt template

## Step1: Design system prompt with instructions
system_prompt = """You are an assistant for question-answering tasks based on PDF documents.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer based on the context, just say that you don't know.
Keep the answer concise but informative.

Context: {context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

## Step 3: Print the prompt template
print(f"System Prompt:\n{system_prompt}")

System Prompt:
You are an assistant for question-answering tasks based on PDF documents.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer based on the context, just say that you don't know.
Keep the answer concise but informative.

Context: {context}



### Build the Complete RAG Chain

In [30]:
# TODO: Build RAG chain using LangChain Expression Language (LCEL)
# 1. Create a format_docs function to combine documents
# 2. Build chain with retriever | prompt | llm | parser
# 3. Print confirmation that chain is created

## Step 1: Create a format_docs function to combine documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


## Step 2: Build chain with retriever | prompt | llm | parser
rag_chain = (
    {
        "context": (lambda x: x["input"]) | retriever | format_docs,
        "input": lambda x: x["input"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

## Step 3. Print confirmation that chain is created
print("Chain is created!")

Chain is created!


1. Takes the user question
2. Send it to the retriever
3. get relevent docuemnt chunks
4. Convert them into clean text

### Test the RAG Chain

In [33]:
# TODO: Ask a question using the RAG chain
# 1. Create a question about the PDF
# 2. Invoke the chain with the question
# 3. Display the answer
# 4. Show the retrieved context separately

## Step 1. Create a question about the PDF:
question = "What is this document about? Give me a breif summary"

## Invoke the chain
answer = rag_chain.invoke({"input": question})

# 3. Display the answer
print(f"Question: {question}\n")
print(f"Answer: \n{answer}")

## Step 4. Show the retrieved context separately
retrieved_docs = retriever.invoke(question)
print(f"\n\nRetrieved {len(retrieved_docs)} relevent_chunks")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n --Chunk {i} (Page {doc.metadata.get('page', 'N/A')})---")
    print(doc.page_content[:200] + '---')

Question: What is this document about? Give me a breif summary

Answer: 
This document is related to an HDFC Life Insurance Policy. It includes information such as the Policy Schedule, Premium Receipt, Terms & Conditions, Service Options, Deletion Details, Salary Updation Details, and Change in Policy-Member Details. It emphasizes the importance of carefully reviewing the information, keeping the Policy Bond safe for availing benefits, and providing necessary details in case of any changes or deletions. The document also mentions the Certificate of Insurance for insured members and provides instructions for communication in the case of an employer-employee relationship.


Retrieved 3 relevent_chunks

 --Chunk 1 (Page 1)---
purchased HDFC Life Insurance Policy:  
 
 Policy Schedule   :  Summary of key features of your HDFC Life Insurance Policy 
 Premium Receipt   :  Acknowledgement of the first Premium paid by you 
 ---

 --Chunk 2 (Page 26)---
above, else provide us the details sep

## Part 7: Interactive Chat Interface

### Build a "Chat with PDF" System
Now let's create an interactive chat interface where users can ask multiple questions!

In [34]:
# TODO: Create chat function
# 1. Create a function that runs a loop for user input
# 2. Accept questions until user types 'quit', 'exit', or 'q'
# 3. Call the RAG chain for each question
# 4. Display the answer and source information
# 5. Handle errors gracefully

def chat_with_pdf(rag_chain, retriever):
    print("PDF Chat Assistant - Ans me anything about your Document!")
    print("Type 'quit', 'exit', 'q' to end our conversation" )

    while True:
        # Get the User Input
        question = input("You: ").strip()

        # Check for exit commands:
        if question.lower() in ['quit', 'exit', 'q', '']:
            print("\n Thanks for chatting! Goodbye!")
            break

        # Get the response for RAG:
        print("\n Thinking")
        try:
            answer = rag_chain.invoke({"input": question})
            print(f"Assistant: {answer}\n")

            context_docs = retriever.invoke(question)
            print(f"Source Found in {len(context_docs)} relevent section")
        except Exception as e:
            print(f"Error {str(e)}\n")



 
 
# To run chat:
# chat_with_pdf(rag_chain, retriever)

In [35]:
chat_with_pdf(rag_chain, retriever)

PDF Chat Assistant - Ans me anything about your Document!
Type 'quit', 'exit', 'q' to end our conversation

 Thinking
Assistant: This document is a Policy Document related to an HDFC Life Insurance Policy. It includes information about benefits, benefit expiry age, certificate of insurance, notices by the company under the policy, the entire contract, and details on the purchased HDFC Life Insurance Policy such as the policy schedule, premium receipt, terms & conditions, and service options. It also provides guidance on the importance of keeping the policy bond safe for availing policy benefits and advises communication of insurance details to employees in case of an employer-employee relationship.

Source Found in 3 relevent section

 Thinking
Assistant: The toll-free number mentioned in the document is 1800 266 9777.

Source Found in 3 relevent section

 Thinking
Assistant: The HDFC Life Group Term Life Insurance policy is a non-linked, non-participating Group Life Insurance policy. 

1. Small Chunks:
- FAQ's
- Short Answer

2. Meduim (300-600):
- Blog
- Documentation

3. Large (600-1000+)
- Legal documents
- Research paper

### Alternative: Batch Q&A Testing

In [ ]:
# TODO: Test with multiple questions
# 1. Create a list of test questions
# 2. Loop through each question
# 3. Get answer from RAG chain
# 4. Display results

test_questions = [
    # Add your test questions here
]

# Write your code here

## 🎓 Homework & Next Steps

### Challenges (Pick One):

1. **Multi-PDF Support** (Medium)
   - Load and query multiple PDFs simultaneously
   - Track which PDF each answer came from

2. **Citation System** (Hard)
   - Make the system cite specific pages/sections
   - Format: "(See page 5, paragraph 2)"

3. **Conversational Memory** (Medium-Hard)
   - Remember previous questions
   - Handle follow-up questions: "Tell me more about that"

4. **UI Development** (Easy)
   - Build a Streamlit interface
   - Allow users to upload PDFs
   - Create a chat-like experience

5. **Semantic Chunking** (Hard)
   - Split at semantic boundaries instead of fixed sizes
   - Detect paragraph/topic changes

### Resources:
- 📚 LangChain Documentation: https://python.langchain.com
- 🔍 FAISS Documentation: https://github.com/facebookresearch/faiss
- 📊 LangSmith: https://smith.langchain.com

### Next Session Preview:
- RAG Optimization Techniques
- Custom Embeddings
- Hybrid Search (Vector + Keyword)
- Evaluation Metrics for RAG

---

## 🎉 Congratulations!

You now understand the complete RAG pipeline! This foundation scales to:
- 1M+ documents
- Multiple users simultaneously
- Mission-critical applications
- Production deployments

**Great work today! 🚀**